# 00 - Chuẩn bị model

Mỗi "cấu hình" là một model ở một mức nén (quantization):
- **F16** = bản gốc, không nén (chính xác nhất, nặng nhất). Riêng Gemma-3 dùng định dạng BF16 (cùng độ chính xác, nhưng không bị lỗi tràn số như F16)
- **Q8** = nén nhẹ (gần như không mất chất lượng), **Q4** = nén mạnh (nhỏ nhất, nhanh nhất, có thể kém chính xác hơn)

Các model nhỏ (dưới 1 tỉ tham số) cố ý chỉ chạy bản không nén, vì đã nhỏ sẵn và nén thêm dễ làm giảm chất lượng.

Tất cả các bản nén đều được tạo từ **cùng một bản gốc** trên Hugging Face, để so sánh công bằng.

**Cần chuẩn bị 1 lần:**
1. `pip install "llama-cpp-python[server]" huggingface_hub psutil requests pandas numpy matplotlib lm_eval datasets`
2. Tải `llama.cpp` về thư mục `llama.cpp/` (để có `convert_hf_to_gguf.py`) và đặt các chương trình `llama-quantize`, `llama-perplexity` vào `llama.cpp/build/bin/` (trên Windows tải bản build sẵn từ trang Releases của llama.cpp).
3. Với Gemma-3: chấp nhận điều khoản trên Hugging Face rồi chạy `huggingface-cli login`.

In [3]:
import slm_tools as st, model_catalog
configs = st.all_configs()
print(len(configs), 'cấu hình')
for c in configs:
    print(('có   ' if c['file'].exists() else 'THIẾU'), c['label'])

15 cấu hình
có    SmolLM2-135M-Instruct (F16)
có    Qwen2.5-0.5B-Instruct (F16)
có    Qwen3-0.6B (F16)
THIẾU Gemma-3-1B-it (F16)
THIẾU Arch-Agent-1.5B (F16)
THIẾU xLAM-2-1B-fc-r (F16)
THIẾU Qwen3-1.7B (F16)
THIẾU Arch-Agent-3B (F16)
THIẾU Arch-Agent-1.5B (Q8)
THIẾU xLAM-2-1B-fc-r (Q8)
THIẾU Qwen3-1.7B (Q8)
THIẾU Arch-Agent-3B (Q8)
THIẾU Phi-4-mini-instruct (Q8)
THIẾU Arch-Agent-3B (Q4)
THIẾU Phi-4-mini-instruct (Q4)


Ô dưới tải model gốc, đổi sang định dạng GGUF và nén. Có thể mất khá lâu và tốn nhiều ổ đĩa (vài chục GB); chạy lại được, file nào đã có sẽ được bỏ qua.

In [4]:
for c in configs:
    if not c['file'].exists():
        print('đang tạo', c['label'])
        model_catalog.ensure_gguf(c['name'], c['quant'], llama_cpp_dir='./llama.cpp')

đang tạo Gemma-3-1B-it (F16)


Fetching 8 files: 100%|██████████| 8/8 [01:01<00:00,  7.65s/it]


[run] c:\Users\Minor\Downloads\slm\.venv\Scripts\python.exe llama.cpp\convert_hf_to_gguf.py model_cache\hf\Gemma-3-1B-it --outfile model_cache\gguf\Gemma-3-1B-it.F16.gguf --outtype bf16
đang tạo Arch-Agent-1.5B (F16)


Fetching 8 files: 100%|██████████| 8/8 [01:15<00:00,  9.42s/it]


[run] c:\Users\Minor\Downloads\slm\.venv\Scripts\python.exe llama.cpp\convert_hf_to_gguf.py model_cache\hf\Arch-Agent-1.5B --outfile model_cache\gguf\Arch-Agent-1.5B.F16.gguf --outtype f16
đang tạo xLAM-2-1B-fc-r (F16)


Fetching 8 files: 100%|██████████| 8/8 [01:16<00:00,  9.55s/it]


... (3 outputs left out)

Fetching 10 files: 100%|██████████| 10/10 [02:25<00:00, 14.54s/it]


[run] c:\Users\Minor\Downloads\slm\.venv\Scripts\python.exe llama.cpp\convert_hf_to_gguf.py model_cache\hf\Arch-Agent-3B --outfile model_cache\gguf\Arch-Agent-3B.F16.gguf --outtype f16
đang tạo Arch-Agent-1.5B (Q8)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Arch-Agent-1.5B.F16.gguf model_cache\gguf\Arch-Agent-1.5B.Q8_0.gguf Q8_0
đang tạo xLAM-2-1B-fc-r (Q8)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\xLAM-2-1B-fc-r.F16.gguf model_cache\gguf\xLAM-2-1B-fc-r.Q8_0.gguf Q8_0
đang tạo Qwen3-1.7B (Q8)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Qwen3-1.7B.F16.gguf model_cache\gguf\Qwen3-1.7B.Q8_0.gguf Q8_0
đang tạo Arch-Agent-3B (Q8)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Arch-Agent-3B.F16.gguf model_cache\gguf\Arch-Agent-3B.Q8_0.gguf Q8_0
đang tạo Phi-4-mini-instruct (Q8)


Fetching 11 files: 100%|██████████| 11/11 [03:12<00:00, 17.47s/it]


[run] c:\Users\Minor\Downloads\slm\.venv\Scripts\python.exe llama.cpp\convert_hf_to_gguf.py model_cache\hf\Phi-4-mini-instruct --outfile model_cache\gguf\Phi-4-mini-instruct.F16.gguf --outtype f16
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Phi-4-mini-instruct.F16.gguf model_cache\gguf\Phi-4-mini-instruct.Q8_0.gguf Q8_0
đang tạo Arch-Agent-3B (Q4)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Arch-Agent-3B.F16.gguf model_cache\gguf\Arch-Agent-3B.Q4_K_M.gguf Q4_K_M
đang tạo Phi-4-mini-instruct (Q4)
[run] llama.cpp\build\bin\Release\llama-quantize.exe model_cache\gguf\Phi-4-mini-instruct.F16.gguf model_cache\gguf\Phi-4-mini-instruct.Q4_K_M.gguf Q4_K_M


**Lưu ý:** F16 của Arch-Agent-3B (~6 GB), Q8 của Arch-Agent-3B (~3,3 GB) và Q8 của Phi-4-mini (~4 GB) không vừa 4 GB RAM của UNO Q. Chúng vẫn được đo trên máy tính làm mốc chuẩn; trên board, kết quả "hết RAM" là một kết quả hợp lệ.